# Evaluate Jantzen cell values

Compare one zeroshot run with the Jantzen ground truth, one cell at a time.

Records are matched inside each paper on source taxon, interaction type, target taxon, and part of target when either side has a part. A pair is also tried with the source and target reversed and the interaction replaced by its inverse. Locality and year do not create a match. After a pair is locked, each field uses its own scorer. An empty prediction CSV is a real zero-record result, so those ground-truth rows stay in the miss count.

The notebook writes `cells.csv`, `fields.csv`, and `papers.csv` under the run's `eval` directory. It does not call the model.

In [ ]:
import sys
from html import escape
from pathlib import Path

from IPython.display import HTML, display

PROJECT_ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'pyproject.toml').exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Open this notebook from inside the project directory.')
sys.path.insert(0, str(PROJECT_ROOT))

from src.eval.jantzen import evaluate_run, write_evaluation


In [ ]:
PREDICTION_DIR = PROJECT_ROOT / 'outputs/zeroshot/jantzen_qwen36_27b_fp8_rules_v2'
INTERACTION_CSV = PROJECT_ROOT / 'data/source/Jantzen_species_interaction/interaction_data.csv'
PAPERS_CSV = PROJECT_ROOT / 'data/source/Jantzen_species_interaction/papers_searched.csv'
OUTPUT_DIR = PREDICTION_DIR / 'eval'

evaluation = evaluate_run(PREDICTION_DIR, INTERACTION_CSV, PAPERS_CSV)
written = write_evaluation(evaluation, OUTPUT_DIR)
print(f'{len(evaluation.papers)} papers')
print(f'Cells: {written["cells"]}')
print(f'Fields: {written["fields"]}')
print(f'Papers: {written["papers"]}')


In [ ]:
FIELD_LABELS = {
    'source_taxon_name': 'Source taxon',
    'interaction_type_name': 'Interaction',
    'target_taxon_name': 'Target taxon',
    'part_of_target_taxon': 'Part of target',
    'time_of_occurrence': 'Season',
    'locality_name': 'Locality',
    'decimal_latitude': 'Latitude',
    'decimal_longitude': 'Longitude',
    'observation_date_time': 'Years',
    'remarks': 'Remarks',
    'evidence_type': 'Evidence type',
}
HEADERS = [
    'Field', 'Both present', 'Prediction only', 'Ground truth only', 'Both missing',
    'Similarity', 'Factual rate', 'Retrieval rate',
]


def show_table(headers, rows):
    head = ''.join(f'<th style="text-align:left;padding:4px 8px">{escape(header)}</th>' for header in headers)
    body = []
    for row in rows:
        body.append(
            '<tr>' + ''.join(
                f'<td style="padding:4px 8px;border-top:1px solid #ccc">{escape(cell)}</td>'
                for cell in row
            ) + '</tr>'
        )
    display(HTML(
        '<table style="border-collapse:collapse">'
        f'<thead><tr>{head}</tr></thead><tbody>{ "".join(body) }</tbody></table>'
    ))


field_rows = evaluation.field_rows()
show_table(
    HEADERS,
    [
        [
            FIELD_LABELS.get(row['field'], row['field']),
            row['both_present'],
            row['prediction_only'],
            row['ground_truth_only'],
            row['both_missing'],
            row['similarity'] or '—',
            row['factual_rate'] or '—',
            row['retrieval_rate'] or '—',
        ]
        for row in field_rows
    ],
)


Similarity is the mean decision score on cells filled on both sides. A wrong label that both sides filled lowers similarity and still counts as both present. Factual rate is both-present / (both-present + prediction-only). Retrieval rate is both-present / (both-present + ground-truth-only). Both-missing cells are counted and left out of those three numbers.

Evidence type is one cell per paper, taken from `papers_searched.csv`. The other fields are interaction rows.

In [ ]:
papers = sorted(
    evaluation.papers,
    key=lambda paper: (
        float(paper.retrieval_rate) if paper.retrieval_rate else -1,
        -paper.n_ground_truth_only,
    ),
)
show_table(
    ['Paper', 'Ground truth', 'Predicted', 'Matched', 'Extra predictions', 'Missed rows', 'Factual', 'Retrieval'],
    [
        [
            paper.paper_id,
            str(paper.n_ground_truth),
            str(paper.n_predicted),
            str(paper.n_matched),
            str(paper.n_prediction_only),
            str(paper.n_ground_truth_only),
            paper.factual_rate or '—',
            paper.retrieval_rate or '—',
        ]
        for paper in papers[:15]
    ],
)
print('Fifteen papers with the lowest record retrieval. The full list is papers.csv.')


In [ ]:
PAPER_ID = '275'
detail = [
    cell for cell in evaluation.cells
    if cell.paper_id == PAPER_ID and cell.gt_row == '1' and cell.pred_record_index
]
if not detail:
    detail = [cell for cell in evaluation.cells if cell.paper_id == PAPER_ID][:11]
show_table(
    ['Field', 'Outcome', 'Ground truth', 'Predicted', 'Score', 'Soft', 'Reversed'],
    [
        [
            FIELD_LABELS.get(cell.field, cell.field),
            cell.outcome,
            cell.ground_truth_value or '—',
            cell.predicted_value or '—',
            cell.decision_score or '—',
            cell.soft_score or '—',
            cell.oriented or '—',
        ]
        for cell in detail
    ],
)
print(
    f'Paper {PAPER_ID}, first matched ground-truth row when one exists. '
    'Reversed 1 means the predicted source and target were swapped before scoring.'
)
